In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# CAN Bus Dataset Analysis\n",
    "## HCRL Car Hacking Dataset - Exploratory Data Analysis\n",
    "\n",
    "This notebook analyzes the HCRL Car Hacking Dataset to understand:\n",
    "- Normal CAN traffic patterns\n",
    "- Attack characteristics\n",
    "- Message distributions\n",
    "- Timing patterns"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "import os\n",
    "\n",
    "# Add src directory to path\n",
    "sys.path.append('../src')\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from dataset_loader import HCRLDataset\n",
    "from utils import Visualizer\n",
    "\n",
    "# Set style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (15, 6)\n",
    "\n",
    "print(\"✓ Libraries imported successfully\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Datasets"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize dataset loader\n",
    "# CHANGE THIS PATH to your dataset location\n",
    "DATASET_PATH = '/Users/harshithmadhavaram/Desktop/can_security_framework/datasets/CAN-intrusion-dataset'\n",
    "\n",
    "loader = HCRLDataset(DATASET_PATH)\n",
    "\n",
    "# Load all datasets\n",
    "print(\"Loading datasets...\\n\")\n",
    "normal_df = loader.load_dataset('normal')\n",
    "dos_df = loader.load_dataset('dos')\n",
    "fuzzy_df = loader.load_dataset('fuzzy')\n",
    "impersonation_df = loader.load_dataset('impersonation')\n",
    "\n",
    "print(\"\\n✓ All datasets loaded\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Dataset Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Print detailed statistics for each dataset\n",
    "for dataset_type in ['normal', 'dos', 'fuzzy', 'impersonation']:\n",
    "    loader.print_statistics(dataset_type)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. CAN ID Distribution Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare CAN ID distributions across datasets\n",
    "fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n",
    "fig.suptitle('CAN ID Distribution Across Datasets', fontsize=16, fontweight='bold')\n",
    "\n",
    "datasets = [\n",
    "    ('normal', normal_df, 'green'),\n",
    "    ('dos', dos_df, 'red'),\n",
    "    ('fuzzy', fuzzy_df, 'orange'),\n",
    "    ('impersonation', impersonation_df, 'purple')\n",
    "]\n",
    "\n",
    "for idx, (name, df, color) in enumerate(datasets):\n",
    "    ax = axes[idx // 2, idx % 2]\n",
    "    \n",
    "    # Get top 15 CAN IDs\n",
    "    top_ids = df['CAN_ID'].value_counts().head(15)\n",
    "    \n",
    "    # Plot\n",
    "    ax.bar(range(len(top_ids)), top_ids.values, color=color, alpha=0.7, edgecolor='black')\n",
    "    ax.set_xticks(range(len(top_ids)))\n",
    "    ax.set_xticklabels([f'0x{id:03X}' for id in top_ids.index], rotation=45, ha='right')\n",
    "    ax.set_title(f'{name.upper()} Dataset', fontsize=14, fontweight='bold')\n",
    "    ax.set_xlabel('CAN ID', fontsize=12)\n",
    "    ax.set_ylabel('Message Count', fontsize=12)\n",
    "    ax.grid(axis='y', alpha=0.3)\n",
    "    \n",
    "    # Add percentage labels on bars\n",
    "    total = df['CAN_ID'].count()\n",
    "    for i, v in enumerate(top_ids.values):\n",
    "        percentage = (v / total) * 100\n",
    "        ax.text(i, v, f'{percentage:.1f}%', ha='center', va='bottom', fontsize=8)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/can_id_distribution.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"✓ CAN ID distribution plot saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Message Timing Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze inter-message intervals\n",
    "fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n",
    "fig.suptitle('Inter-Message Timing Distribution', fontsize=16, fontweight='bold')\n",
    "\n",
    "for idx, (name, df, color) in enumerate(datasets):\n",
    "    ax = axes[idx // 2, idx % 2]\n",
    "    \n",
    "    # Calculate inter-message intervals\n",
    "    df_sorted = df.sort_values('Timestamp')\n",
    "    intervals = df_sorted['Timestamp'].diff().dropna() * 1000  # Convert to ms\n",
    "    \n",
    "    # Remove outliers for better visualization (keep 99th percentile)\n",
    "    intervals_filtered = intervals[intervals < np.percentile(intervals, 99)]\n",
    "    \n",
    "    # Plot histogram\n",
    "    ax.hist(intervals_filtered, bins=100, color=color, alpha=0.7, edgecolor='black')\n",
    "    ax.set_title(f'{name.upper()} Dataset', fontsize=14, fontweight='bold')\n",
    "    ax.set_xlabel('Inter-message Interval (ms)', fontsize=12)\n",
    "    ax.set_ylabel('Frequency', fontsize=12)\n",
    "    ax.grid(alpha=0.3)\n",
    "    \n",
    "    # Add statistics text box\n",
    "    stats_text = f\"Mean: {intervals.mean():.4f} ms\\n\"\n",
    "    stats_text += f\"Median: {intervals.median():.4f} ms\\n\"\n",
    "    stats_text += f\"Std: {intervals.std():.4f} ms\"\n",
    "    ax.text(0.65, 0.95, stats_text, transform=ax.transAxes,\n",
    "            fontsize=10, verticalalignment='top',\n",
    "            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/timing_distribution.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"✓ Timing distribution plot saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Message Rate Over Time"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze message rate over time (first 10 seconds of each dataset)\n",
    "fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n",
    "fig.suptitle('Message Rate Over Time (First 10 Seconds)', fontsize=16, fontweight='bold')\n",
    "\n",
    "for idx, (name, df, color) in enumerate(datasets):\n",
    "    ax = axes[idx // 2, idx % 2]\n",
    "    \n",
    "    # Get first 10 seconds\n",
    "    df_sorted = df.sort_values('Timestamp')\n",
    "    start_time = df_sorted['Timestamp'].min()\n",
    "    df_subset = df_sorted[df_sorted['Timestamp'] <= start_time + 10].copy()\n",
    "    \n",
    "    # Calculate message rate in 0.1 second windows\n",
    "    df_subset['Time_Window'] = ((df_subset['Timestamp'] - start_time) / 0.1).astype(int) * 0.1\n",
    "    rate = df_subset.groupby('Time_Window').size()\n",
    "    \n",
    "    # Plot\n",
    "    ax.plot(rate.index, rate.values, color=color, linewidth=2, alpha=0.7)\n",
    "    ax.fill_between(rate.index, rate.values, alpha=0.3, color=color)\n",
    "    ax.set_title(f'{name.upper()} Dataset', fontsize=14, fontweight='bold')\n",
    "    ax.set_xlabel('Time (seconds)', fontsize=12)\n",
    "    ax.set_ylabel('Messages per 0.1s', fontsize=12)\n",
    "    ax.grid(alpha=0.3)\n",
    "    \n",
    "    # Add average rate line\n",
    "    avg_rate = rate.mean()\n",
    "    ax.axhline(y=avg_rate, color='black', linestyle='--', linewidth=1, alpha=0.5)\n",
    "    ax.text(0.5, avg_rate, f'Avg: {avg_rate:.1f} msg/0.1s', \n",
    "            fontsize=10, va='bottom')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/message_rate_timeline.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"✓ Message rate timeline plot saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Data Byte Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze data byte distributions\n",
    "fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n",
    "fig.suptitle('Data Byte Value Distribution', fontsize=16, fontweight='bold')\n",
    "\n",
    "for idx, (name, df, color) in enumerate(datasets):\n",
    "    ax = axes[idx // 2, idx % 2]\n",
    "    \n",
    "    # Collect all data bytes\n",
    "    all_bytes = []\n",
    "    for i in range(8):\n",
    "        all_bytes.extend(df[f'Data{i}'].values)\n",
    "    \n",
    "    # Plot distribution\n",
    "    ax.hist(all_bytes, bins=256, range=(0, 256), color=color, alpha=0.7, edgecolor='black')\n",
    "    ax.set_title(f'{name.upper()} Dataset', fontsize=14, fontweight='bold')\n",
    "    ax.set_xlabel('Byte Value (0-255)', fontsize=12)\n",
    "    ax.set_ylabel('Frequency', fontsize=12)\n",
    "    ax.grid(alpha=0.3)\n",
    "    \n",
    "    # Add entropy calculation\n",
    "    from scipy.stats import entropy\n",
    "    hist, _ = np.histogram(all_bytes, bins=256, range=(0, 256))\n",
    "    probabilities = hist / np.sum(hist)\n",
    "    data_entropy = entropy(probabilities + 1e-10, base=2)\n",
    "    \n",
    "    ax.text(0.65, 0.95, f'Entropy: {data_entropy:.4f} bits', \n",
    "            transform=ax.transAxes, fontsize=12, verticalalignment='top',\n",
    "            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/data_byte_distribution.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"✓ Data byte distribution plot saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Attack Characteristics Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create summary comparison table\n",
    "summary_data = []\n",
    "\n",
    "for name, df, _ in datasets:\n",
    "    df_sorted = df.sort_values('Timestamp')\n",
    "    intervals = df_sorted['Timestamp'].diff().dropna() * 1000\n",
    "    \n",
    "    # Collect data bytes\n",
    "    all_bytes = []\n",
    "    for i in range(8):\n",
    "        all_bytes.extend(df[f'Data{i}'].values)\n",
    "    \n",
    "    # Calculate entropy\n",
    "    hist, _ = np.histogram(all_bytes, bins=256, range=(0, 256))\n",
    "    probabilities = hist / np.sum(hist)\n",
    "    data_entropy = entropy(probabilities + 1e-10, base=2)\n",
    "    \n",
    "    summary_data.append({\n",
    "        'Dataset': name.upper(),\n",
    "        'Total Messages': len(df),\n",
    "        'Unique IDs': df['CAN_ID'].nunique(),\n",
    "        'Avg Interval (ms)': intervals.mean(),\n",
    "        'Median Interval (ms)': intervals.median(),\n",
    "        'Data Entropy (bits)': data_entropy,\n",
    "        'Duration (s)': df['Timestamp'].max() - df['Timestamp'].min(),\n",
    "        'Avg Rate (msg/s)': len(df) / (df['Timestamp'].max() - df['Timestamp'].min())\n",
    "    })\n",
    "\n",
    "summary_df = pd.DataFrame(summary_data)\n",
    "\n",
    "# Display table\n",
    "print(\"\\n\" + \"=\"*100)\n",
    "print(\"DATASET COMPARISON SUMMARY\")\n",
    "print(\"=\"*100)\n",
    "print(summary_df.to_string(index=False))\n",
    "print(\"=\"*100)\n",
    "\n",
    "# Save to CSV\n",
    "summary_df.to_csv('../results/dataset_summary.csv', index=False)\n",
    "print(\"\\n✓ Summary saved to dataset_summary.csv\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Key Observations\n",
    "\n",
    "Based on the analysis above, we can observe:\n",
    "\n",
    "### Normal Traffic:\n",
    "- Consistent message rates\n",
    "- Predictable CAN ID distribution\n",
    "- Regular timing intervals\n",
    "- Lower data entropy\n",
    "\n",
    "### DoS Attack:\n",
    "- Very high message rate\n",
    "- Dominated by single CAN ID\n",
    "- Very short inter-message intervals\n",
    "- Repetitive data patterns\n",
    "\n",
    "### Fuzzing Attack:\n",
    "- High data entropy\n",
    "- Random byte values\n",
    "- Variable message rates\n",
    "- Targeting specific IDs\n",
    "\n",
    "### Impersonation Attack:\n",
    "- Spoofs legitimate IDs\n",
    "- May have unusual timing\n",
    "- Data may differ from legitimate sources\n",
    "\n",
    "These characteristics will guide our ML model development in the next notebook."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Save Analysis Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save datasets for next notebook\n",
    "import pickle\n",
    "\n",
    "datasets_dict = {\n",
    "    'normal': normal_df,\n",
    "    'dos': dos_df,\n",
    "    'fuzzy': fuzzy_df,\n",
    "    'impersonation': impersonation_df\n",
    "}\n",
    "\n",
    "with open('../results/processed_datasets.pkl', 'wb') as f:\n",
    "    pickle.dump(datasets_dict, f)\n",
    "\n",
    "print(\"✓ Processed datasets saved for ML training\")\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"DATASET ANALYSIS COMPLETE\")\n",
    "print(\"=\"*60)\n",
    "print(\"\\nNext: Open 02_ml_model_training.ipynb to train detection models\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}